In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Preprocessing
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib  # for saving encoders/scalers

print(f"TensorFlow version: {tf.__version__}")
print(" All libraries loaded successfully")

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

TensorFlow version: 2.20.0
 All libraries loaded successfully


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# Clone repo
os.chdir('/content')
!git clone https://github.com/VAL-Jerono/Dynamic_Pricing.git 2>/dev/null || \
(cd Dynamic_Pricing && git pull)

# Repo code
os.chdir('/content/Dynamic_Pricing')
sys.path.insert(0, 'src')

# Data path
DATA_PATH = "/content/drive/MyDrive/DYNAMIC PRICING/DATA SETS"



data = pd.read_csv("/content/drive/MyDrive/DYNAMIC PRICING/DATA SETS/FLIGHTS DATA SETS/flight_data_enriched_usd.csv")
data.head(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,duration_category,flight_date,season,jet_fuel_price_inr_per_kl,distance_km,exchange_rate_usd_inr,jet_fuel_price_usd_per_kl
0,SpiceJet,1408,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
1,SpiceJet,1387,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,Long,2022-02-12,High,86038,1148.09,75.341,1141.98
2,AirAsia,1213,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
3,Vistara,1559,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
4,Vistara,1549,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,Long,2022-02-12,High,86038,1148.09,75.341,1141.98
5,Vistara,1541,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.33,1,5955,Long,2022-02-12,High,86038,1148.09,75.341,1141.98
6,Vistara,1533,Delhi,Morning,zero,Morning,Mumbai,Economy,2.08,1,6060,Short,2022-02-12,High,86038,1148.09,75.341,1141.98
7,Vistara,1543,Delhi,Afternoon,zero,Evening,Mumbai,Economy,2.17,1,6060,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
8,GO_FIRST,1013,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.17,1,5954,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
9,GO_FIRST,1014,Delhi,Afternoon,zero,Evening,Mumbai,Economy,2.25,1,5954,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98


In [4]:
#data = pd.read_csv("/content/drive/MyDrive/flight_data_cleaned.csv")

print(f"Shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
data.head(10)

Shape: (300153, 18)

Columns: ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'duration', 'days_left', 'price', 'duration_category', 'flight_date', 'season', 'jet_fuel_price_inr_per_kl', 'distance_km', 'exchange_rate_usd_inr', 'jet_fuel_price_usd_per_kl']


,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,duration_category,flight_date,season,jet_fuel_price_inr_per_kl,distance_km,exchange_rate_usd_inr,jet_fuel_price_usd_per_kl
0,SpiceJet,1408,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
1,SpiceJet,1387,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,Long,2022-02-12,High,86038,1148.09,75.341,1141.98
2,AirAsia,1213,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
3,Vistara,1559,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
4,Vistara,1549,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,Long,2022-02-12,High,86038,1148.09,75.341,1141.98
5,Vistara,1541,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.33,1,5955,Long,2022-02-12,High,86038,1148.09,75.341,1141.98
6,Vistara,1533,Delhi,Morning,zero,Morning,Mumbai,Economy,2.08,1,6060,Short,2022-02-12,High,86038,1148.09,75.341,1141.98
7,Vistara,1543,Delhi,Afternoon,zero,Evening,Mumbai,Economy,2.17,1,6060,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
8,GO_FIRST,1013,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.17,1,5954,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
9,GO_FIRST,1014,Delhi,Afternoon,zero,Evening,Mumbai,Economy,2.25,1,5954,Medium,2022-02-12,High,86038,1148.09,75.341,1141.98
